## Task 1.1: Core Contribution / Architecture

**Paper:** *Finding Deceptive Opinion Spam by Any Stretch of the Imagination*  
**Authors:** Myle Ott, Yejin Choi, Claire Cardie, Jeffrey T. Hancock  
**Venue:** ACL 2011  

---

## Step-by-Step Method Description

---

### Step 1: Gold-Standard Dataset Construction

- **Description:** The authors construct the first publicly available labeled dataset of deceptive opinion spam. They collect 400 *truthful* positive hotel reviews by scraping 5-star reviews from TripAdvisor for the 20 most popular Chicago hotels, filtering out first-time reviewers and short reviews. They collect 400 *deceptive* reviews by posting Human Intelligence Tasks (HITs) on Amazon Mechanical Turk (AMT), asking Turkers to write convincing fake positive reviews for those same hotels.
- **Reference:** Section 3 (Dataset Construction), Table 1 (descriptive statistics of deceptive submissions)
- **Purpose:** Prior work on opinion spam detection had no gold-standard labeled data, forcing researchers to use heuristics such as detecting duplicate reviews (Jindal and Liu, 2008). This step establishes a reliable 800-review benchmark (400 truthful + 400 deceptive) with known ground truth, enabling meaningful supervised classification and evaluation.

---

### Step 2: Feature Extraction via Three Independent Framings

The authors extract three distinct feature sets from each review, each motivated by a different theoretical framing of what deception looks like in text:

#### Step 2a: Genre Identification Features (POS Tags)
- **Description:** Each review is parsed using the Stanford Parser (Klein and Manning, 2003) to obtain the relative frequency of every Part-of-Speech (POS) tag (e.g., noun singular, verb past tense, personal pronoun). This produces one numeric feature per POS tag for each review.
- **Reference:** Section 4.1 (Genre Identification)
- **Purpose:** Based on Rayson et al. (2001), truthful writing tends to be *informative* (more nouns, prepositions, determiners) while deceptive writing tends to be *imaginative* (more verbs, pronouns, adverbs). POS frequencies capture this genre-level signal without looking at specific words.

#### Step 2b: Psycholinguistic Features (LIWC)
- **Description:** Each review is processed by the LIWC2007 software (Pennebaker et al., 2007), which maps review text onto 80 psychologically meaningful category scores — including emotional processes, spatial references, social references, and cognitive processes. This produces an 80-dimensional feature vector per review.
- **Reference:** Section 4.2 (Psycholinguistic Deception Detection)
- **Purpose:** Psychological deception research (Hancock et al., 2008; Newman et al., 2003) suggests liars show characteristic linguistic signatures — increased negative emotion, psychological distancing, and reduced spatial detail. LIWC features capture these cross-domain deception cues.

#### Step 2c: Text Categorization Features (n-grams)
- **Description:** Each review is represented using bag-of-words n-gram features: UNIGRAMS (single words), BIGRAMS+ (unigrams + bigrams), and TRIGRAMS+ (unigrams + bigrams + trigrams). All features are lowercased and unstemmed. Document vectors are normalized to unit length.
- **Reference:** Section 4.3 (Text Categorization), Equation 3
- **Purpose:** Unlike POS or LIWC features, n-gram features capture both the *content* and *context* of what is written. They allow the classifier to learn which specific words and phrases are statistically predictive of deceptive versus truthful writing in this domain.

---

### Step 3: Classifier Training — Naïve Bayes and Linear SVM

- **Description:** Two classifier families are trained on each feature set. The **Naïve Bayes (NB)** classifier uses the language model formulation from Zhou et al. (2008), estimating per-class language models using the SRI Language Modeling Toolkit with Kneser-Ney smoothing. The **Support Vector Machine (SVM)** classifier uses SVMlight (Joachims, 1999) to learn a linear weight vector **w** and bias *b*, classifying a document **x** by the sign of **w · x + b** (Equation 3 in the paper). Document vectors are normalized to unit length before training.
- **Reference:** Section 4.4 (Classifiers), Equations 1, 2, 3
- **Purpose:** SVM and NB are chosen because both have established strong performance on text classification tasks (Joachims, 1998; Sebastiani, 2002). Linear SVMs are specifically preferred here because they produce interpretable weight vectors, enabling the feature analysis in Section 5 that reveals *which linguistic signals* are most predictive of deception.

---

### Step 4: Nested 5-Fold Cross-Validation Evaluation

- **Description:** All classifiers are evaluated using a nested 5-fold cross-validation (CV) procedure. Critically, folds are constructed so that **each fold contains all reviews from exactly four hotels** — meaning the model is always tested on reviews from hotels it has never seen during training. Hyperparameters are selected for each test fold via standard CV on the training folds.
- **Reference:** Section 5 (Results and Discussion), Table 3
- **Purpose:** Splitting by hotel (rather than randomly by review) prevents data leakage — a model trained on reviews from the same hotel it is tested on could learn hotel-specific writing styles rather than general deception signals. This design ensures the evaluation measures generalisation to unseen hotels.

---

### Step 5: Feature Weight Analysis and Theoretical Interpretation

- **Description:** After training, the weight vectors **w** learned by the linear SVM classifiers are examined to identify which features most strongly predict deceptive or truthful reviews. The top 15 highest-weighted features for each class are reported for LIWC+BIGRAMS+SVM and LIWCSVM (Table 5). The POS weights are compared against genre-theoretic predictions from Rayson et al. (2001) (Table 4).
- **Reference:** Section 5 (Results and Discussion), Tables 4 and 5
- **Purpose:** This step moves beyond classification performance to provide *theoretical contributions*. The feature analysis reveals that truthful reviews use more spatial and specific language ("bathroom", "location", "floor") while deceptive reviews focus on external context ("husband", "vacation", "business") — a finding consistent with reality monitoring theory (Johnson and Raye, 1981) and liars' difficulty encoding spatial information (Vrij et al., 2009).

---

### Step 6: Human Performance Benchmarking

- **Description:** Three undergraduate student volunteers evaluate 160 reviews (one cross-validation fold) and label each as truthful or deceptive. Their performance is compared against all automated classifiers. Two meta-judges are also evaluated: MAJORITY (predicts deceptive if ≥2 judges agree) and SKEPTIC (predicts deceptive if any judge agrees).
- **Reference:** Section 3.3 (Human Performance), Table 2
- **Purpose:** This benchmarking serves two functions. First, it validates the dataset — if humans cannot distinguish the reviews, the deceptive reviews are genuinely convincing. Second, it contextualises the classifier performance: the best SVM achieves 89.8% accuracy while the best human judge achieves only 61.9%, demonstrating the clear value of the automated approach.

---

## Final Summary Sentence

This paper solves the problem of automatically detecting deceptive hotel reviews written to deceive human readers, and the authors claim their approach is better than existing alternatives because it is the first to compare n-gram text categorization, psycholinguistic (LIWC), and genre-based (POS) feature sets on a gold-standard labeled dataset with genuine deceptive opinions, demonstrating that n-gram–based linear SVM classifiers achieve nearly 90% accuracy — far beyond the ≈60% ceiling of human judges and the heuristic methods of prior work.